# Schwarzman Scholars Open Data — Full Control Overview
**1,497 scholars · 74 public intros · 285 bios · 4 archetypes · Warmth v2 + Whisper**

This notebook oversees *everything* we simulated — no black box. Run top-to-bottom for full control. All paths are relative to the repo root (`..` from `notebooks/`).

> **Run:** `pip install -r ../requirements.txt` (or `pandas matplotlib scikit-learn nltk`) then `jupyter lab notebooks/schwarzman_overview.ipynb`

Shorthand legend: `data/video_legend.csv` → `AA18` = Abdullah Almiqasbi 2018 (First+Last+cohort). Use it as the axis label on every PNG so 74 names stay readable.


In [ ]:
# --- 0) Setup ---
import pathlib, sys
ROOT = pathlib.Path("..").resolve()  # notebooks/ -> repo root
DATA = ROOT / "data"
AD = ROOT / "analytics_dashboard"
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# check files
for p in [DATA/"schwarzman_scholars_dataset.csv", DATA/"video_legend.csv", DATA/"bios_clusters.csv", AD/"schwarzman_hybrid_4K.png"]:
    print("✓" if p.exists() else "✗", p.relative_to(ROOT), p.stat().st_size if p.exists() else "")


## 1) The dataset at a glance — who gets in (no ML)
Raw facts: 1497 rows, 74 videos (4.9%), 285 bios (19%). This is what survives bias.


In [ ]:
df = pd.read_csv(DATA/"schwarzman_scholars_dataset.csv", encoding="utf-8")
print(f"rows {len(df)} | videos {df['has_intro_video'].sum()} | bios {df['bio'].notna().sum()}")
df["has_intro_video"] = pd.to_numeric(df["has_intro_video"], errors="coerce").fillna(0).astype(int)
# top countries / unis — same as README
print(df["country"].value_counts().head(8).to_string())
print(df["university"].value_counts().head(8).to_string())
# show the hybrid treemap (the image you shared)
from IPython.display import Image, display
try:
    display(Image(filename=str(AD/"schwarzman_hybrid_4K.png"), width=800))
except: 
    print("Hybrid treemap at analytics_dashboard/schwarzman_hybrid_4K.png — open it for USA/China/global/policy blocks")


## 2) Bios language — 4 archetypes (public sklearn TF-IDF + KMeans)
We cluster 285 bios into 4 doors. No private model — just `pandas + scikit-learn + nltk`.
- TF-IDF 400 words (stopwords + `the/and` stripped, `and 1412 > the 1129` would otherwise dominate)
- KMeans k=4, PCA 2D for `bios_clusters_pca.png`
- Shorthand `AA18` etc. for dots (First+Last, readable at 300 dpi)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

bios = df[df["bio"].notna()].copy()
STOP = ["a","an","the","and","or","but","if","while","of","at","by","for","with","about","into","through","during","before","after","above","below","to","from","up","down","in","out","on","off","over","under","again","further","then","once","here","there","when","where","why","how","all","any","both","each","few","more","most","other","some","such","no","nor","not","only","own","same","so","than","too","very","s","t","can","will","just","don","should","now","is","are","was","were","be","been","being","has","have","had","do","does","did"]
try:
    from nltk.corpus import stopwords
    import nltk
    try: nltk.data.find("corpora/stopwords")
    except: nltk.download("stopwords", quiet=True)
    STOP = list(set(STOP) | set(stopwords.words("english")))
except: pass

vec = TfidfVectorizer(stop_words=STOP, max_features=400, token_pattern=r"[a-z']+", lowercase=True)
X = vec.fit_transform(bios["bio"].astype(str))
kmeans = KMeans(n_clusters=4, random_state=7, n_init=20)
labels = kmeans.fit_predict(X)
bios["cluster"] = labels
bios["shorthand"] = bios["name"].apply(lambda n: (n.split()[0][0]+n.split()[-1][0]).upper() if len(n.split())>1 else n[:2].upper())

terms = vec.get_feature_names_out()
order = kmeans.cluster_centers_.argsort()[:, ::-1]
names = {0:"Health Systems (n27)", 1:"Climate & China Bridge (n70)", 2:"Policy/International (n84)", 3:"Tech-Business Builders (n104)"}
for ki in range(4):
    top = [terms[i] for i in order[ki, :10]]
    print(f"Cluster {ki}: {names[ki]} — top {', '.join(top)} — n={(labels==ki).sum()}")

# PCA plot (saved as analytics_dashboard/bios_clusters_pca.png)
pca = PCA(n_components=2, random_state=7)
X2 = pca.fit_transform(X.toarray())
bios["pca0"], bios["pca1"] = X2[:,0], X2[:,1]

plt.figure(figsize=(12,7))
colors = ["#ef4444","#10b981","#0ea5e9","#f59e0b"]
for ki in range(4):
    m = bios["cluster"]==ki
    plt.scatter(bios.loc[m,"pca0"], bios.loc[m,"pca1"], c=colors[ki], label=f"{ki}: {names[ki]}", s=70, alpha=0.78, edgecolor="white")
    for _, r in bios[m].sample(2, random_state=ki).iterrows():
        plt.annotate(r["shorthand"], (r["pca0"], r["pca1"]), fontsize=7, weight="bold",
                     bbox=dict(facecolor="white", edgecolor=colors[ki], boxstyle="round,pad=0.2", alpha=0.85))
plt.title("4 archetypes — PCA of 285 bios (TF-IDF 400 + KMeans)")
plt.xlabel("PCA 1"); plt.ylabel("PCA 2"); plt.legend(); plt.grid(alpha=0.12); plt.tight_layout()
plt.savefig(AD/"bios_clusters_pca.png", dpi=300, bbox_inches="tight"); plt.close()
print("saved bios_clusters_pca.png")

# also show pre-saved cluster sizes
try:
    display(Image(filename=str(AD/"bios_cluster_sizes.png"), width=700))
    display(Image(filename=str(AD/"bios_clusters_by_region.png"), width=700))
except: pass


**Name distributions grouped by cluster** — each is a door you can copy. Full lists in `data/cluster_names/cluster_*.csv`.
- Tech-Business `n104` 36%: `AZ Adele Zhong (CN), AD Akorfa Dagadu (Ghana/MIT)…` — US 38%·CN 15% dispersed
- Policy/Intl `n84` 29%: `AH Aili Hou (US/Columbia), AG Aleena Gul (US/Yale)…` — US 64% Harvard-heavy
- Climate-Bridge `n70` 25%: `AS Ajay Sawant (India), LO Lozangtashi (CN/Tibet)…` — CN 44%
- Health `n27` 9%: `AB Anita Bassey (US), DC Daphne Chebesi (Cameroon)…` — most dispersed, Africa 15%


In [ ]:
# Grouped name tables — read before you write your own bio
import pathlib
clusters = pd.read_csv(DATA/"bios_clusters.csv", encoding="utf-8")
for ki in sorted(clusters["cluster"].unique()):
    sub = clusters[clusters["cluster"]==ki].sort_values("name")
    print(f"\n--- Cluster {ki}: {names[ki]} ---")
    # join with region for context
    print(sub.merge(pd.read_csv(DATA/"bios_clusters_with_region.csv", encoding="utf-8")[["name","region"]], on="name").groupby("region").size().to_string())
    # show 10 shorthands
    print("shorthands:", ", ".join(sub["shorthand"].head(10).tolist()))
    # preview first 5 full names
    display(sub[["shorthand","name","country","university","cohort_year"]].head(5))


## 3) All 74 videos — shorthand + where Schwarzman plays vs itself / vs others
`data/video_legend.csv` gives `AA18..XR27` for every intro (73 unique vids, `SL26`/`SL27` share one). Use `shorthand` on every axis so plots stay readable.
- Warmth v2 = `happy*0.9 + neutral*0.25 - fear*0.15 - sad*0.10 +35` (7 frames, fallback Retina→OpenCV→MTCNN) — mean 63.1 (n=24 scored, 50 pending) vs old +30 hack 71.2
- Sentiment = TextBlob on Whisper tiny (>0.10 optimistic) — videos 0.14 vs bios 0.064 flat


In [ ]:
legend = pd.read_csv(DATA/"video_legend.csv", encoding="utf-8")
feat = pd.read_csv(DATA/"video_features_all.csv", encoding="utf-8")
print(f"legend {len(legend)} rows — {legend['warmth_v2'].notna().sum()} scored, {legend['warmth_v2'].isna().sum()} pending")
print(legend.head(5).to_string(index=False))
# region share vs all 1497
all_regions = df["country"].map(lambda c: "US" if c=="United States of America" else "CN" if c=="China" else "Other").value_counts(normalize=True).round(3)
print("All 1497 US/CN/Other share:", all_regions.to_dict())
print("YT 74 region share:", legend["region"].value_counts(normalize=True).round(3).to_dict())
# cohort share
print(legend["cohort"].value_counts().sort_index().to_string())
# show all74 shorthand plots we pre-generated
try:
    display(Image(filename=str(AD/"warmth_vs_sentiment_all74_shorthand.png"), width=900))
    display(Image(filename=str(AD/"charisma_by_cohort_all74_shorthand.png"), width=900))
except: print("plots at analytics_dashboard/*all74_shorthand.png")


### How Schwarzman plays
- **vs itself (video sharers vs all 1497):** US 44.6% vs 41.2% even, **CN 2.7% vs 20.0% ↓ 0.7% share** (Chinese scholars rarely post YouTube), Europe 11.8% / LatAm 10.8% over-share — any charisma finding is Western-skewed.
- **vs other fellowships:** no external dump, so we use `region` — the US/CN bridge *is* Schwarzman (61% US+CN vs Rhodes general). Filter `region=="CN"` or `cluster==1` to see the bridge door.


In [ ]:
# One-liner to make your own plot with shorthand
import matplotlib.pyplot as plt
# join bios cluster + video warmth via shorthand/name
merged = pd.merge(pd.read_csv(DATA/"bios_clusters.csv", encoding="utf-8"), legend, left_on="name", right_on="name", how="inner", suffixes=("_bio","_vid"))
# example: PCA colored by warmth for the 24 who have both
has_both = merged[merged["warmth_v2"].notna()]
print(f"bios+video overlap: {len(has_both)} scholars have both a bio and a scored video")
if len(has_both)>0:
    # we need pca coords
    pca_df = pd.read_csv(DATA/"bios_clusters_pca.csv", encoding="utf-8")
    has_both = has_both.merge(pca_df[["name","pca0","pca1"]], on="name")
    plt.figure(figsize=(10,6))
    sc = plt.scatter(has_both["pca0"], has_both["pca1"], c=pd.to_numeric(has_both["warmth_v2"], errors="coerce"), cmap="Blues", s=100, edgecolor="white")
    plt.colorbar(sc, label="Warmth v2")
    for _, r in has_both.iterrows():
        plt.annotate(r["shorthand_vid"] if "shorthand_vid" in r else r["shorthand_bio"], (r["pca0"], r["pca1"]), fontsize=7, weight="bold")
    plt.title("Where warmth lives in the 4 doors (bios PCA + video warmth)")
    plt.xlabel("PCA 1"); plt.ylabel("PCA 2"); plt.tight_layout()
    plt.savefig(AD/"custom_pca_warmth_shorthand.png", dpi=300, bbox_inches="tight")
    print("saved custom_pca_warmth_shorthand.png")
    display(Image(filename=str(AD/"custom_pca_warmth_shorthand.png"), width=800))


## 4) What they actually accept — 5 hypotheses + how to sway your fit
See `docs/HYPOTHESES.md` for the full write-up. TL;DR:

- **H1 Four doors, not one** — Tech 36%, Policy 29%, Climate 25%, Health 9% (stable 2026 vs 2027). Don’t blend doors — hybrids blur in PCA middle.
- **H2 Bridge > brilliance** — `china/global/international` top-10 in *every* cluster. Your SOP needs one US↔China translation moment (Geneva-style).
- **H3 Verb > noun** — `founded 73, led 71` beat `passionate 58`. Show 3 verbs with numbers.
- **H4 University as signal, not filter** — Harvard loads only in Policy; Tech/Climate are feeder-dispersed (Fresno `CS27`, Montana `DM27` win there).
- **H5 Video as compensatory** — Warm Africa/LatAm over-share 10-14% vs CN 0.7%; if you’re under-represented, a 60s YouTube intro is leverage.

**To sway:** filter `data/bios_clusters_with_region.csv` where `cluster==your door` and `region==your region`, read 5 bios in that slice — copy *sentence shape*, not content. Then `python scripts/bios_nlp.py` on your draft to see which cluster you land in.


In [ ]:
# --- Validate the repo (what generate_plots.py and validate.py do) ---
import subprocess, sys
print("Running scripts/validate.py...")
try:
    out = subprocess.run([sys.executable, str(ROOT/"scripts/validate.py")], capture_output=True, text=True, timeout=30)
    print(out.stdout); print(out.stderr)
except Exception as e:
    print("validate err", e)

print("\nDocs:")
for p in [ROOT/"docs/HYPOTHESES.md", ROOT/"docs/VIDEO_LEGEND.md", ROOT/"README.md"]:
    print(p.relative_to(ROOT), p.stat().st_size)


## Next — fill the 50 pending videos (free Whisper)
```bash
# all 74, checkpointed per-video to data/meta/*.json + data/transcripts/*.txt
python scripts/video_pipeline.py            # skip existing (24)
python scripts/video_pipeline.py --force    # re-score with fear/sad penalty
make transcripts  # alias
# then re-run this notebook top-to-bottom — legend fills NA → scores, all74 plots auto-complete
```

The hybrid 4K treemap you shared (`USA/China/global/...`) and the L-system tree remain the visual essay — the 4 clusters are the *why* behind those big blocks. Use `shorthand` on every new PNG you make so reviewers can join back to `data/video_legend.csv` in one glance.
